In [0]:
# Databricks notebook source

# 01 - Bronze Ingestion

Ingestão incremental simulada dos dados de corridas de táxi da NYC TLC.
Cada execução deste notebook processa **um mês** específico, definido pelos widgets `year` e `month`.
A carga é idempotente: reprocessar o mesmo mês não gera duplicidade, graças ao `MERGE INTO` baseado
em um hash de linha. Uma tabela de controle (`_ingestion_log`) registra o histórico de execuções.
Fonte: https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page


Configuração e widgets

In [0]:
dbutils.widgets.text("year", "2024", "Ano (YYYY)")
#dbutils.widgets.text("month", "01", "Mês (MM)")
 
year = dbutils.widgets.get("year")
#month = dbutils.widgets.get("month")
months = [f"{m:02d}" for m in range(1, 13)]

#year_month = f"{year}-{month}"
 
CATALOG = "nyc_taxi"
BRONZE_SCHEMA = "bronze"
VOLUME_PATH = f"/Volumes/{CATALOG}/{BRONZE_SCHEMA}/raw_files"
 
BRONZE_TABLE = f"{CATALOG}.{BRONZE_SCHEMA}.trips_bronze"
INGESTION_LOG_TABLE = f"{CATALOG}.{BRONZE_SCHEMA}._ingestion_log"
 
#SOURCE_URL = f"https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_{year_month}.parquet"
#LOCAL_FILE_PATH = f"{VOLUME_PATH}/yellow_tripdata_{year_month}.parquet"
 
#print(f"Processando mês: {year_month}")
#print(f"Origem: {SOURCE_URL}")
#print(f"Destino no volume: {LOCAL_FILE_PATH}")

print(f"Ano a processar: {year}")
print(f"Meses: {months}")

## Criação das tabelas de controle (idempotente)
Roda uma vez por ambiente novo; `CREATE TABLE IF NOT EXISTS` garante que reexecuções não quebram nada.

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {BRONZE_TABLE} (
    row_hash STRING,
    year_month STRING,
    VendorID INT,
    tpep_pickup_datetime TIMESTAMP,
    tpep_dropoff_datetime TIMESTAMP,
    passenger_count DOUBLE,
    trip_distance DOUBLE,
    RatecodeID DOUBLE,
    store_and_fwd_flag STRING,
    PULocationID INT,
    DOLocationID INT,
    payment_type BIGINT,
    fare_amount DOUBLE,
    extra DOUBLE,
    mta_tax DOUBLE,
    tip_amount DOUBLE,
    tolls_amount DOUBLE,
    improvement_surcharge DOUBLE,
    total_amount DOUBLE,
    congestion_surcharge DOUBLE,
    airport_fee DOUBLE,
    _source_file STRING,
    _ingested_at TIMESTAMP
)
USING DELTA
PARTITIONED BY (year_month)
""")
 
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {INGESTION_LOG_TABLE} (
    year_month STRING,
    status STRING,
    rows_processed BIGINT,
    source_file STRING,
    started_at TIMESTAMP,
    finished_at TIMESTAMP
)
USING DELTA
""")
 
print("Tabelas prontas.")

## Funções auxiliares

A lógica de ingestão de um mês (antes era o corpo inteiro do notebook) virou uma função Python,
chamada uma vez por mês dentro do loop principal mais abaixo.
 

In [0]:
import requests
from datetime import datetime
from pyspark.sql import functions as F
 
 
def file_exists_in_volume(path: str) -> bool:
    try:
        dbutils.fs.ls(path)
        return True
    except Exception:
        return False
 
 
def already_ingested(year_month: str) -> bool:
    return (
        spark.table(INGESTION_LOG_TABLE)
        .filter((F.col("year_month") == year_month) & (F.col("status") == "SUCCESS"))
        .count()
        > 0
    )
 
 
def log_result(year_month, status, rows_processed, source_file, started_at, finished_at):
    log_row = spark.createDataFrame(
        [(year_month, status, rows_processed, source_file, started_at, finished_at)],
        schema="year_month STRING, status STRING, rows_processed BIGINT, source_file STRING, started_at TIMESTAMP, finished_at TIMESTAMP",
    )
    log_row.write.mode("append").saveAsTable(INGESTION_LOG_TABLE)
 
 
def ingest_month(year: str, month: str) -> str:
    """Processa um único mês. Retorna 'SUCCESS', 'SKIPPED' ou 'FAILED'."""
    year_month = f"{year}-{month}"
    started_at = datetime.now()
 
    if already_ingested(year_month):
        print(f"[{year_month}] já ingerido com sucesso, pulando.")
        return "SKIPPED"
 
    source_url = f"https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_{year_month}.parquet"
    local_file_path = f"{VOLUME_PATH}/yellow_tripdata_{year_month}.parquet"
 
    if not file_exists_in_volume(local_file_path):
        try:
            print(f"[{year_month}] baixando de {source_url} ...")
            response = requests.get(source_url, timeout=60)
            response.raise_for_status()
            with open(local_file_path, "wb") as f:
                f.write(response.content)
            print(f"[{year_month}] download concluído.")
        except Exception as e:
            print(f"[{year_month}] falha no download: {e}")
            print(
                f"[{year_month}] provável bloqueio de rede do Free Edition. "
                f"Faça upload manual do arquivo em {VOLUME_PATH} e rode este notebook novamente."
            )
            log_result(year_month, "FAILED", 0, local_file_path, started_at, datetime.now())
            return "FAILED"
    else:
        print(f"[{year_month}] arquivo já presente no volume, pulando download.")
 
    if not file_exists_in_volume(local_file_path):
        log_result(year_month, "FAILED", 0, local_file_path, started_at, datetime.now())
        return "FAILED"
 
    try:
        raw_df = spark.read.parquet(local_file_path)
        business_columns = raw_df.columns
 
        df = (
            raw_df
            .withColumn(
                "row_hash",
                F.sha2(F.concat_ws("||", *[F.col(c).cast("string") for c in business_columns]), 256),
            )
            .withColumn("year_month", F.lit(year_month))
            .withColumn("_source_file", F.lit(local_file_path))
            .withColumn("_ingested_at", F.current_timestamp())
        )
 
        row_count = df.count()
        df.createOrReplaceTempView("staged_trips")
 
        spark.sql(f"""
        MERGE INTO {BRONZE_TABLE} AS target
        USING staged_trips AS source
        ON target.row_hash = source.row_hash AND target.year_month = source.year_month
        WHEN NOT MATCHED THEN INSERT *
        """)
 
        finished_at = datetime.now()
        log_result(year_month, "SUCCESS", row_count, local_file_path, started_at, finished_at)
        print(f"[{year_month}] concluído: {row_count} linhas.")
        return "SUCCESS"
 
    except Exception as e:
        print(f"[{year_month}] erro inesperado no processamento: {e}")
        log_result(year_month, "FAILED", 0, local_file_path, started_at, datetime.now())
        return "FAILED"

## Loop principal — processa os 12 meses do ano

In [0]:
results = {}
 
for month in months:
    year_month = f"{year}-{month}"
    print(f"\n=== {year_month} ===")
    results[year_month] = ingest_month(year, month)

## Resumo da execução

In [0]:
success = [k for k, v in results.items() if v == "SUCCESS"]
skipped = [k for k, v in results.items() if v == "SKIPPED"]
failed = [k for k, v in results.items() if v == "FAILED"]
 
print(f"Sucesso: {len(success)} — {success}")
print(f"Pulados (já ingeridos): {len(skipped)} — {skipped}")
print(f"Falhas: {len(failed)} — {failed}")
 
if failed:
    dbutils.notebook.exit(f"Ingestão concluída com falhas em: {failed}")
else:
    dbutils.notebook.exit(f"Ingestão do ano {year} concluída com sucesso ({len(success)} novos, {len(skipped)} já existentes).")

## Verificação rápida

In [0]:
display(spark.table(INGESTION_LOG_TABLE).orderBy("year_month"))